In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
DS_FOLDER = "downsampled_full_fov_128x128x64_crop-17.5"

splits_df = pd.read_csv(_PROJECT_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Non-skipped patients: {len(patients_df)}")

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Non-skipped patients: 209


In [2]:
VZ_FLOW_TAG = 5  # tag_0x0043_0x1030 == 5 is vz

results = []

for _, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    patient_dir = PATIENT_DATA_DIR / pid

    catalog_path = patient_dir / f"dicom_catalog_{pid}.csv"
    if not catalog_path.exists():
        print(f"SKIP {pid}: no DICOM catalog")
        results.append({"patient_id": pid, "split": split, "original_direction": "NO_CATALOG"})
        continue

    catalog = pd.read_csv(catalog_path)

    # Filter to 4D flow vz component
    vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
    if len(vz_cat) == 0:
        print(f"SKIP {pid}: no vz component in catalog")
        results.append({"patient_id": pid, "split": split, "original_direction": "NO_VZ"})
        continue

    # Compute time_index and slice_index from instance numbers (original acquisition order)
    vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
    vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]

    # Extract z-coordinate from ImagePositionPatient
    vz_cat["ipp"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x)))
    vz_cat["z"] = vz_cat["ipp"].apply(lambda x: x[2])

    # Look at time_index 0, sorted by slice_index (original acquisition order)
    t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("slice_index")
    z_values = t0["z"].values

    # Determine original direction
    z_diff = np.diff(z_values)
    n_increasing = np.sum(z_diff > 0)
    n_decreasing = np.sum(z_diff < 0)

    if n_increasing > n_decreasing:
        orig_dir = "I_to_S"  # z increases with slice_index
    elif n_decreasing > n_increasing:
        orig_dir = "S_to_I"  # z decreases with slice_index
    else:
        orig_dir = "AMBIGUOUS"

    # Compute slice normal from ImageOrientationPatient
    iop = vz_cat["imageorientation"].iloc[0]
    iop_arr = np.array(eval(iop))
    row_vec = iop_arr[:3]
    col_vec = iop_arr[3:]
    slice_normal = np.cross(row_vec, col_vec)
    dot_z = slice_normal[2]  # dot product with z-hat

    results.append({
        "patient_id": pid,
        "split": split,
        "original_direction": orig_dir,
        "n_slices": len(t0),
        "z_range": f"[{z_values.min():.1f}, {z_values.max():.1f}]",
        "z_first_slice": z_values[0],
        "z_last_slice": z_values[-1],
        "slice_normal_z": dot_z,
        "n_increasing": n_increasing,
        "n_decreasing": n_decreasing,
    })

results_df = pd.DataFrame(results)
print("\n=== Summary ===")
print(results_df["original_direction"].value_counts())
print(f"\nPatients originally S→I (needed flip): {(results_df['original_direction'] == 'S_to_I').sum()}")
print(f"Patients originally I→S (no flip):      {(results_df['original_direction'] == 'I_to_S').sum()}")


=== Summary ===
original_direction
I_to_S    117
S_to_I     92
Name: count, dtype: int64

Patients originally S→I (needed flip): 92
Patients originally I→S (no flip):      117


In [3]:
results_df

,patient_id,split,original_direction,n_slices,z_range,z_first_slice,z_last_slice,slice_normal_z,n_increasing,n_decreasing
0,Balboloop,test,I_to_S,140,"[-89.5, 160.7]",-89.5459,160.6540,0.999930,139,0
1,Biswifo,test,S_to_I,120,"[-211.9, -21.5]",-21.4812,-211.8810,0.999966,0,119
2,Bomatog,test,S_to_I,140,"[-124.9, 97.5]",97.4641,-124.9360,1.000040,0,139
3,Boochuto,test,S_to_I,120,"[-116.5, 73.9]",73.9028,-116.4970,0.999930,0,119
4,Boumorim,test,S_to_I,120,"[-87.7, 102.7]",102.6960,-87.7036,0.999966,0,119
...,...,...,...,...,...,...,...,...,...,...
204,Somenut,validation,I_to_S,152,"[6.8, 218.2]",6.8000,218.2000,0.999930,151,0
205,Suedrurnep,validation,I_to_S,140,"[-100.0, 122.5]",-99.9500,122.4500,0.999942,139,0
206,Swibihek,validation,I_to_S,120,"[144.3, 334.7]",144.3060,334.7070,1.000040,119,0
207,Ugitat,validation,S_to_I,140,"[-99.2, 95.4]",95.4250,-99.1751,1.000000,0,139


In [11]:
from scipy.stats import pearsonr

COMPONENTS = [
    ("4d_flow_vx", "4d_flow_vx_corr", "Vx"),
    ("4d_flow_vy", "4d_flow_vy_corr", "Vy"),
    ("4d_flow_vz", "4d_flow_vz_corr", "Vz"),
]

analysis_rows = []

for _, res in results_df.iterrows():
    pid = res["patient_id"]
    orig_dir = res["original_direction"]

    if orig_dir in ("NO_CATALOG", "NO_VZ"):
        continue

    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DS_FOLDER
    row_data = {"patient_id": pid, "original_direction": orig_dir}

    for subfolder, corr_subfolder, label in COMPONENTS:
        uncorr_path = ds_root / subfolder / f"{subfolder}_{pid}_frame_00.nii.gz"
        corr_path = ds_root / corr_subfolder / f"{corr_subfolder}_{pid}_frame_00.nii.gz"

        if not uncorr_path.exists() or not corr_path.exists():
            row_data[f"{label}_r"] = np.nan
            row_data[f"{label}_sign"] = "MISSING"
            continue

        uncorr = nib.load(str(uncorr_path)).get_fdata(dtype=np.float32).ravel()
        corr = nib.load(str(corr_path)).get_fdata(dtype=np.float32).ravel()

        r, _ = pearsonr(uncorr, corr)
        r_neg, _ = pearsonr(uncorr, -corr)
        row_data[f"{label}_r"] = r
        row_data[f"{label}_r_neg"] = r_neg
        row_data[f"{label}_sign"] = "SAME" if r > 0 else "NEGATED"

    analysis_rows.append(row_data)
    print(f"{pid} done")

analysis_df = pd.DataFrame(analysis_rows)
print("\nDone.")

Balboloop done
Biswifo done
Bomatog done
Boochuto done
Boumorim done
Bovutou done
Cadotueg done
Detodu done
Diecudey done
Diepami done
Diequipi done
Dithigog done
Dublafer done
Dujomal done
Elagieg done
Golotag done
Grequafie done
Gueshifa done
Kuquelok done
Oduskueb done
Quetode done
Runusath done
Sepigoo done
Stonscuetof done
Suquepog done
Tercippun done
Tiepolem done
Tisupey done
Amifer done
Aruborn done
Asonlig done
Badiswu done
Bibathot done
Bogeebo done
Boudubat done
Burapo done
Butiswu done
Cadedag done
Cefaru done
Cemuquey done
Ceriba done
Ceyebum done
Coosimo done
Cornuefor done
Crutaswo done
Dalibul done
Dapafem done
Datokif done
Desoomi done
Diboscey done
Difresa done
Dinaspig done
Drusdinut done
Dudoblo done
Duquestank done
Dutungub done
Edengat done
Egumud done
Ekotey done
Emalem done
Epcedin done
Ernegur done
Erusar done
Eyostoy done
Fibrefob done
Fliesiemo done
Frahidiel done
Fudoquo done
Fuekeswa done
Fumtufoos done
Geedefou done
Gepeta done
Getahig done
Gidusi done
Gif

In [12]:
print("=" * 80)
print("SIGN CHECK: is corrected velocity negated relative to uncorrected?")
print("  SAME   = positive correlation → signs match (correct)")
print("  NEGATED = negative correlation → one is incorrectly negated")
print("=" * 80)

for label in ["Vx", "Vy", "Vz"]:
    col = f"{label}_sign"
    valid = analysis_df[analysis_df[col] != "MISSING"]
    negated_mask = valid[col] == "NEGATED"
    n_negated = negated_mask.sum()
    print(f"\n{label}: {n_negated}/{len(valid)} patients have NEGATED sign")
    if n_negated > 0:
        negated_patients = valid[negated_mask]
        by_dir = negated_patients.groupby("original_direction").size()
        print(f"  Breakdown by original direction: {dict(by_dir)}")

print("\n")
print("=" * 80)
print("CROSS-TAB: Original direction vs sign, per component")
print("=" * 80)
for label in ["Vx", "Vy", "Vz"]:
    col = f"{label}_sign"
    valid = analysis_df[analysis_df[col] != "MISSING"]
    print(f"\n--- {label} ---")
    print(pd.crosstab(valid["original_direction"], valid[col]))

print("\n")
print("=" * 80)
print("EXPECTED vs ACTUAL")
print("  I→S patients: vz should be negated once → corrected & uncorrected should match (SAME)")
print("  S→I patients: vz should be double-negated (=not negated) → if code only negated once,")
print("                corrected & uncorrected will be NEGATED (wrong sign)")
print("=" * 80)
vz_valid = analysis_df[analysis_df["Vz_sign"] != "MISSING"]
print(pd.crosstab(vz_valid["original_direction"], vz_valid["Vz_sign"], margins=True))

SIGN CHECK: is corrected velocity negated relative to uncorrected?
  SAME   = positive correlation → signs match (correct)
  NEGATED = negative correlation → one is incorrectly negated

Vx: 0/208 patients have NEGATED sign

Vy: 0/208 patients have NEGATED sign

Vz: 0/208 patients have NEGATED sign


CROSS-TAB: Original direction vs sign, per component

--- Vx ---
Vx_sign             SAME
original_direction      
I_to_S               116
S_to_I                92

--- Vy ---
Vy_sign             SAME
original_direction      
I_to_S               116
S_to_I                92

--- Vz ---
Vz_sign             SAME
original_direction      
I_to_S               116
S_to_I                92


EXPECTED vs ACTUAL
  I→S patients: vz should be negated once → corrected & uncorrected should match (SAME)
  S→I patients: vz should be double-negated (=not negated) → if code only negated once,
                corrected & uncorrected will be NEGATED (wrong sign)
Vz_sign             SAME  All
original_direc

In [13]:
display_cols = [
    "patient_id", "original_direction",
    "Vx_r", "Vx_r_neg", "Vx_sign",
    "Vy_r", "Vy_r_neg", "Vy_sign",
    "Vz_r", "Vz_r_neg", "Vz_sign",
]
with pd.option_context("display.max_rows", None, "display.float_format", "{:.4f}".format):
    display(analysis_df[display_cols].sort_values(["original_direction", "Vz_sign", "patient_id"]))

,patient_id,original_direction,Vx_r,Vx_r_neg,Vx_sign,Vy_r,Vy_r_neg,Vy_sign,Vz_r,Vz_r_neg,Vz_sign
42,Coosimo,I_to_S,NaN,NaN,MISSING,NaN,NaN,MISSING,NaN,NaN,MISSING
29,Aruborn,I_to_S,0.9174,-0.9174,SAME,0.9157,-0.9157,SAME,0.8449,-0.8449,SAME
30,Asonlig,I_to_S,0.9154,-0.9154,SAME,0.9059,-0.9059,SAME,0.9089,-0.9089,SAME
31,Badiswu,I_to_S,0.9579,-0.9579,SAME,0.9377,-0.9377,SAME,0.9636,-0.9636,SAME
0,Balboloop,I_to_S,0.9276,-0.9276,SAME,0.9066,-0.9066,SAME,0.9254,-0.9254,SAME
32,Bibathot,I_to_S,0.9859,-0.9859,SAME,0.9933,-0.9933,SAME,0.9891,-0.9891,SAME
34,Boudubat,I_to_S,0.9878,-0.9878,SAME,0.9943,-0.9943,SAME,0.9853,-0.9853,SAME
5,Bovutou,I_to_S,0.9948,-0.9948,SAME,0.9964,-0.9964,SAME,0.9942,-0.9942,SAME
36,Butiswu,I_to_S,0.9575,-0.9575,SAME,0.9811,-0.9811,SAME,0.9623,-0.9623,SAME
37,Cadedag,I_to_S,0.9490,-0.9490,SAME,0.9838,-0.9838,SAME,0.9394,-0.9394,SAME


In [4]:
OUTPUT_DIR = Path("velocity_direction_check_images")
OUTPUT_DIR.mkdir(exist_ok=True)


def load_vol(path: Path) -> np.ndarray:
    return nib.load(str(path)).get_fdata(dtype=np.float32)


for i, (_, res) in enumerate(results_df.iterrows()):
    pid = res["patient_id"]
    orig_dir = res["original_direction"]

    if orig_dir in ("NO_CATALOG", "NO_VZ"):
        continue

    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DS_FOLDER
    vz_path = ds_root / "4d_flow_vz" / f"4d_flow_vz_{pid}_frame_00.nii.gz"
    vz_corr_path = ds_root / "4d_flow_vz_corr" / f"4d_flow_vz_corr_{pid}_frame_00.nii.gz"
    diff_vz_path = ds_root / "4d_flow_diff_vz" / f"4d_flow_diff_vz_{pid}_frame_00.nii.gz"

    if not vz_path.exists() or not vz_corr_path.exists():
        print(f"SKIP {pid}: missing vz or vz_corr")
        continue

    vz = load_vol(vz_path)
    vz_corr = load_vol(vz_corr_path)
    has_diff = diff_vz_path.exists()
    diff_vz = load_vol(diff_vz_path) if has_diff else vz_corr - vz

    n_slices = vz.shape[2]
    mid_z = n_slices // 2
    slices_to_show = [0, n_slices // 4, mid_z, 3 * n_slices // 4, n_slices - 1]

    n_cols = len(slices_to_show)
    fig, axes = plt.subplots(3, n_cols, figsize=(3.5 * n_cols, 9))

    needed_flip = "YES" if orig_dir == "S_to_I" else "NO"
    fig.suptitle(
        f"{pid}  |  Original: {orig_dir}  |  Needed flip: {needed_flip}  |  "
        f"slice_normal_z: {res.get('slice_normal_z', 'N/A'):.3f}",
        fontsize=12, fontweight="bold", y=1.02,
    )

    row_labels = ["Uncorrected Vz", "Corrected Vz", "Diff (corr - uncorr)"]

    for c, si in enumerate(slices_to_show):
        vz_slice = vz[:, :, si].T
        vz_corr_slice = vz_corr[:, :, si].T
        diff_slice = diff_vz[:, :, si].T

        # Symmetric color range for each
        vmax_vz = np.percentile(np.abs(vz), 99)
        vmax_corr = np.percentile(np.abs(vz_corr), 99)
        vmax_diff = np.percentile(np.abs(diff_vz), 99) if np.any(diff_vz != 0) else 1.0

        for r, (slc, vmax) in enumerate([
            (vz_slice, vmax_vz),
            (vz_corr_slice, vmax_corr),
            (diff_slice, vmax_diff),
        ]):
            ax = axes[r, c]
            ax.imshow(slc, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="equal")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(f"z={si}", fontsize=10)
            if c == 0:
                ax.set_ylabel(row_labels[r], fontsize=10)

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{orig_dir}_{pid}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"[{i+1}/{len(results_df)}] {pid} ({orig_dir})")

print(f"\nDone. Images saved to {OUTPUT_DIR.resolve()}")

[1/209] Balboloop (I_to_S)
[2/209] Biswifo (S_to_I)
[3/209] Bomatog (S_to_I)
[4/209] Boochuto (S_to_I)
[5/209] Boumorim (S_to_I)
[6/209] Bovutou (I_to_S)
[7/209] Cadotueg (S_to_I)
[8/209] Detodu (I_to_S)
[9/209] Diecudey (I_to_S)
[10/209] Diepami (S_to_I)
[11/209] Diequipi (S_to_I)
[12/209] Dithigog (S_to_I)
[13/209] Dublafer (I_to_S)
[14/209] Dujomal (S_to_I)
[15/209] Elagieg (I_to_S)
[16/209] Golotag (S_to_I)
[17/209] Grequafie (I_to_S)
[18/209] Gueshifa (I_to_S)
[19/209] Kuquelok (S_to_I)
[20/209] Oduskueb (S_to_I)
[21/209] Quetode (S_to_I)
[22/209] Runusath (S_to_I)
[23/209] Sepigoo (S_to_I)
[24/209] Stonscuetof (S_to_I)
[25/209] Suquepog (I_to_S)
[26/209] Tercippun (S_to_I)
[27/209] Tiepolem (S_to_I)
[28/209] Tisupey (S_to_I)
[29/209] Amifer (S_to_I)
[30/209] Aruborn (I_to_S)
[31/209] Asonlig (I_to_S)
[32/209] Badiswu (I_to_S)
[33/209] Bibathot (I_to_S)
[34/209] Bogeebo (S_to_I)
[35/209] Boudubat (I_to_S)
[36/209] Burapo (S_to_I)
[37/209] Butiswu (I_to_S)
[38/209] Cadedag (I_to_S)